In [ ]:
!pip install gcsfs==2024.6.1
!pip install transformers==4.45.2
!pip install datasets[audio] accelerate evaluate jiwer tensorboard gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
  Attempting uninstall: gcsfs
    Found existing installation: gcsfs 2024.10.0
    Uninstalling gcsfs-2024.10.0:
      Successfully uninstalled gcsfs-2024.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.0
    Uninstalling tokenizers-0.21.0:
      Successfully uninstalled tokenizers-0.21.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1
   ━━━━━━

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copiar el archivo .tar desde Google Drive a Colab
!cp /content/drive/MyDrive/Speech-to-text-translator/Common-Voice-Corpus-4/es.tar /content/

In [ ]:
# Extraer el archivo .tar
!tar -xzf /content/es.tar -C /content/

In [ ]:
# Listar los archivos extraidos
!ls /content/

clips	 drive	 invalidated.tsv  sample_data  train.tsv
dev.tsv  es.tar  other.tsv	  test.tsv     validated.tsv


In [ ]:
# Eliminar archivo .tar
!rm /content/es.tar

In [ ]:
from datasets import load_dataset, DatasetDict

raw_dataset = DatasetDict()

raw_dataset = load_dataset("facebook/covost2", "es_en", data_dir="/content/")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

covost2.py:   0%|          | 0.00/6.96k [00:00<?, ?B/s]

The repository for facebook/covost2 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/covost2.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/79015 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/13221 [00:00<?, ? examples/s]

In [ ]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 79015
    })
    validation: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 13221
    })
    test: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 13221
    })
})

In [ ]:
raw_dataset = raw_dataset.remove_columns(["client_id", "file", "id"])

In [ ]:
test_dataset = raw_dataset["test"]

In [ ]:
test_dataset

Dataset({
    features: ['audio', 'sentence', 'translation'],
    num_rows: 13221
})

In [ ]:
# Use the GPU if available, otherwise fall back to CPU
import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import pipeline

asr_pipe = pipeline(model="Berly00/whisper-large-v3-spanish", device=device)
mt_pipe = pipeline(model="Berly00/mbart-large-50-spanish-to-english", device=device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/992 [00:00<?, ?B/s]

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
bleu_metric = evaluate.load("bleu")

In [ ]:
id = 4
sample = test_dataset[id]["audio"]["path"]
result = asr_pipe(sample, generate_kwargs={"language": "spanish"})["text"]
wer = 100 * wer_metric.compute(predictions=[result], references=[test_dataset[id]["sentence"]])
print(f"WER: {wer:.2f}")
print(f'{result}\n{test_dataset[id]["sentence"]}')

/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


WER: 7.69
En los años que siguieron, este trabajo esparta produjo docenas de buenos jugadores.
En los años que siguieron, este trabajo Esparta produjo docenas de buenos jugadores.


In [ ]:
sample2 = result
result2 = mt_pipe(sample2)[0]["generated_text"]
bleu = 100 * bleu_metric.compute(predictions=[result2], references=[[test_dataset[id]["translation"]]])['bleu']
print(f"BLEU: {bleu:.2f}")
print(f'{result2}\n{test_dataset[id]["translation"]}')

BLEU: 50.78
In the years that followed, this work produced twelve good players.
In the years that followed, this Spartan job produced dozens of good players.


**Metodo 1**

In [ ]:
def process_batch(batch):

    audio_file = batch['audio']['path']
    reference_transcription = batch['sentence']
    reference_translation = batch['translation']

    # Perform ASR
    predicted_transcription = asr_pipe(audio_file, generate_kwargs={"language": "spanish"})["text"]

    # Calculate WER for the batch
    wer = 100 * wer_metric.compute(predictions=[predicted_transcription], references=[reference_transcription])

    # Perform MT
    predicted_translation = mt_pipe(predicted_transcription)[0]['generated_text']

    # Calculate BLEU for the batch
    bleu = 100 * bleu_metric.compute(predictions=[predicted_translation], references=[[reference_translation]])["bleu"]  # Note: references should be a nested list for BLEU

    return {"wer": wer, "bleu": bleu}

In [ ]:
def process_batch(batch):
    audio_files = [sample["path"] for sample in batch["audio"]]
    reference_transcriptions = [sample for sample in batch["sentence"]]
    reference_translations = [sample for sample in batch["translation"]]

    # Perform ASR for the batch
    predicted_transcriptions = asr_pipe(audio_files, generate_kwargs={"language": "spanish"}, batch_size=16)  # Adjust batch_size as needed
    predicted_transcriptions = [result["text"] for result in predicted_transcriptions]  # Extract text from pipeline output

    # Calculate WER for the batch
    wers = [100 * wer_metric.compute(predictions=[pred], references=[ref])
            for pred, ref in zip(predicted_transcriptions, reference_transcriptions)] # Compute WER for each sample and store in a list

    # Perform MT for the batch
    predicted_translations = mt_pipe(predicted_transcriptions, batch_size=16)  # Adjust batch_size as needed
    predicted_translations = [result["generated_text"] for result in predicted_translations]  # Extract translated text

    # Calculate BLEU for the batch
    bleus = [100 * bleu_metric.compute(predictions=[pred], references=[[ref]])["bleu"]
             for pred, ref in zip(predicted_translations, reference_translations)]

    return {"wers": wers, "bleus": bleus}

In [ ]:
result = process_batch(test_dataset[:32])
result

/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'wers': [28.57142857142857,
  11.11111111111111,
  14.285714285714285,
  9.090909090909092,
  7.6923076923076925,
  0.0,
  35.714285714285715,
  90.9090909090909,
  0.0,
  0.0,
  0.0,
  33.33333333333333,
  0.0,
  0.0,
  25.0,
  12.5,
  23.076923076923077,
  0.0,
  0.0,
  0.0,
  9.090909090909092,
  7.142857142857142,
  22.22222222222222,
  0.0,
  7.6923076923076925,
  14.285714285714285,
  9.090909090909092,
  0.0,
  8.333333333333332,
  0.0,
  0.0,
  21.428571428571427],
 'bleus': [0.0,
  0.0,
  0.0,
  0.0,
  50.78431769269641,
  43.66835442847812,
  0.0,
  0.0,
  37.684991644924196,
  0.0,
  51.544589013981735,
  0.0,
  0.0,
  44.28500142691473,
  0.0,
  0.0,
  20.687206010259406,
  0.0,
  0.0,
  0.0,
  0.0,
  100.0,
  25.476965408249004,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  74.87402156832422,
  53.107253497886994,
  50.265872700455304,
  22.718709780542312]}

In [ ]:
results = test_dataset.map(process_batch, batched=True, batch_size=16);

Parameter 'function'=<function process_batch at 0x7dbfdc95f1c0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/13221 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is

In [ ]:
average_wer = sum(results["wers"]) / len(results["wers"])
average_bleu = sum(results["bleus"]) / len(results["bleus"])

print(f"Average WER: {average_wer}")
print(f"Average BLEU: {average_bleu}")

Average WER: 11.263847484482481
Average BLEU: 31.133279902723327


In [ ]:
def split_text(text, chunk_size=225):
    """Splits a text into smaller chunks of a specified size.

    Args:
        text: The input text to split.
        chunk_size: The maximum size of each chunk.

    Returns:
        A list of text chunks.
    """
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])
    return chunks

# Example usage:
large_text = "Your large text here..."
chunks = split_text(large_text)

# Now you can process each chunk individually.

In [ ]:
from transformers import pipeline
import gradio as gr

def translate(audio):

    predicted_transcription = asr_pipe(audio, return_timestamps=True)["text"]

    chunks = split_text(predicted_transcription)

    translated_chunks = []
    for chunk in chunks:
        translated_chunk = mt_pipe(chunk)[0]["generated_text"]
        translated_chunks.append(translated_chunk)

    predicted_translation = " ".join(translated_chunks)

    return predicted_translation

iface = gr.Interface(
    fn=translate,
    inputs=gr.Audio("microfone", type="filepath"),
    outputs="text",
    title="Whisper + MBart",
    description="Realtime demo for Spanish-to-English cascade translation using a fine-tuned Whisper and mBart model.",
)

iface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00e02881588adf098c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
text =  translate('/content/Huida Chad.mp3')
text

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


'Jules, let’s fly! Arruined I am, everyone knows me. I have no escape, maybe I should just fly away. But no, that would not make a chat. Chat, I am right. Yes, I am a chat. Uida, it will be done and no one will stop me, no matter if I am thrown in the trash You know, carnal, this is not the end of Cachifre I still have friends and muscles, I can go out, go back to the Internet, but for now, nobody will defend me Huiracha, Sarae, nobody will defend me. When I talk to you, Cachifre will disappear from youLas words of unknown people and lonely women.'